# Analisis Machine Learning: Stealth Extension Exfiltration (SEE)

Notebook ini memuat _pipeline_ Machine Learning untuk mendeteksi ancaman SEE pada ekstensi browser (Google Chrome). Analisis didasarkan pada metode **Hybrid Analysis (Statis & Dinamis)** dengan menerapkan teknik **K-Fold Cross-Validation** untuk evaluasi model yang lebih kuat dan tidak bias (menggantikan *validation set* tradisional).

*(Jika dijalankan di Google Colab, pastikan Anda telah mengunggah file `dataset.csv` ke dalam folder yang sama dengan notebook ini)*

In [ ]:
# Instalasi library (Jalankan ini jika di Google Colab)
!pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

# Set style untuk visualisasi
sns.set_theme(style="whitegrid")

## 1. Memuat Dataset

In [ ]:
# Memuat dataset hasil ekstraksi Hybrid Analysis
# Sesuaikan path jika dijalankan di Google Colab (misalnya cukup 'dataset.csv')
try:
    df = pd.read_csv('dataset.csv')
except FileNotFoundError:
    df = pd.read_csv('data/features/dataset.csv')

print(f"Total ekstensi dalam dataset: {len(df)}")
print("\nDistribusi Kelas Awal (0 = Benign, 1 = Vulnerable):")
print(df['label'].value_counts())

df.head()

## 2. Preprocessing & Pencegahan Feature Leakage

In [ ]:
# Memisahkan fitur (X) dan target (y)
metadata_cols = ['extension_id', 'see_categories']
y = df['label']

# PENCEGAHAN FEATURE LEAKAGE:
# Fitur has_host_permissions dihapus karena itu adalah definisi klasifikasi SEE,
# bukan observasi perilaku berbahaya. Kita memaksa model untuk belajar dari perilaku.
leakage_cols = ['has_host_permissions', 'host_permissions_count']
X = df.drop(columns=['label'] + metadata_cols + leakage_cols, errors='ignore')
X = X.fillna(0)

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Distribusi Training Set:\n{y_train.value_counts()}")
print("\n(SMOTE tidak digunakan karena distribusi kelas sudah seimbang secara natural)")

## 3. Pelatihan Model (Random Forest + Hyperparameter Tuning + Cross-Validation)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Melatih model...")
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# Parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Menggunakan 5-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Pelatihan Selesai!\nParameter Terbaik: {grid_search.best_params_}")

# Skor CV untuk melihat kestabilan model (menggantikan Validation Set)
cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"\nCV ROC-AUC Scores (5-fold): {cv_scores}")
print(f"Mean CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 4. Evaluasi Model pada Data Uji (Test Set)

In [ ]:
y_pred = best_model.predict(X_test)

print(f"Akurasi: {accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Vulnerable (1)']))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Vulnerable'], yticklabels=['Benign', 'Vulnerable'])
plt.title('Confusion Matrix')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

## 5. Visualisasi Fitur Paling Berpengaruh (Feature Importance)

In [ ]:
importances = best_model.feature_importances_
feature_names = X.columns

# Mengurutkan fitur berdasarkan kepentingannya
feature_df = pd.DataFrame({'Fitur': feature_names, 'Kepentingan': importances})
feature_df = feature_df.sort_values(by='Kepentingan', ascending=False).head(15) # Ambil top 15

plt.figure(figsize=(10, 8))
sns.barplot(x='Kepentingan', y='Fitur', data=feature_df, palette='viridis')
plt.title('Top 15 Fitur Paling Berpengaruh (Feature Importance)')
plt.xlabel('Bobot Kepentingan')
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.show()